# Project 2: Combinatorial Optimization Project

In [15]:
import cvxpy as cp
import numpy as np
import pandas as pd

## Problem statement
Find the optimal time assigning strategy such that we can maximize the learning gain in a week with completing the assignment which is due in the week.

<div align="center">

<figure>
<img src="image/Course calendar.png" height="400" width="500">

<figcaption><em>Figure 1: Sample week schedule.</em></figcaption>

</figure>

</div>

## Variables and Parameters

| Symbol | Description | Type |
|:------:|-------------|:----:|
| $i \in I$ | Index for courses | index |
| $j \in J$ | Index for hourly time slots in a day ($0,\dots,23$) | index |
| $k \in K$ | Index for days of the week (Mon-Sun) | index |
| $x_{ijk}$ | $x_{ijk}=1$ if self-study for course $i$ is scheduled at hour $j$ on day $k$, and $0$ otherwise | decision variable |
| $y_{ijk}$ | $y_{ijk}=1$ if class for course $i$ is attended at hour $j$ on day $k$, and $0$ otherwise | decision variable |
| $z_{ijk}$ | $z_{ijk}=1$ if assignment work for course $i$ is scheduled at hour $j$ on day $k$, and $0$ otherwise | decision variable |
| $s_{jk}$ | $s_{jk}=1$ if rest is scheduled at hour $j$ on day $k$, and $0$ otherwise | decision variable |
| $a_{ijk}$ | $a_{ijk}=1$ if course $i$ has a scheduled class at hour $j$ on day $k$, and $0$ otherwise | parameter |
| $\alpha_i$ | Learning gain from one hour of self-study for course $i$ | parameter |
| $\beta_i$ | Learning gain from one hour of attending class for course $i$ | parameter |
| $\gamma_i$ | Learning gain from one hour of assignment work for course $i$ | parameter |
| $\sigma_i$ | Assignment completion gained from one hour of assignment work for course $i$ | parameter |
| $L_i$ | Minimum number of class hours required per week for course $i$ | parameter |
| $A_i$ | The amount of assignment work for course $i$ need to do to complete the assignment before due day | parameter |
| $D_i$ | The day of due day for course $i$, from 1 to 7 (Monday to Sunday) | parameter |
| $P_i$ | The prerequisite learning gain needed to strat working on assignment for course $i$ | parameter |
| $G_i$ | The maximum learning gain can get for course $i$ | parameter |
| $H$  | The total hours we can scheduled | parameter |

In [16]:
# Sets
courses = ["MATH1", "HISTORY", "MATH2", "EOSC"]
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
hours = list(range(24))

I = range(len(courses))
J = range(len(hours))
K = range(len(days))

nI = len(courses)
nJ = len(hours)
nK = len(days)

course_to_idx = {c: i for i, c in enumerate(courses)}
day_to_idx = {d: k for k, d in enumerate(days)}

# sample parameters
# learning gain per hour (to be changed?)
alpha = np.array([8, 7, 8, 6])   # self-study gain
beta  = np.array([6, 5, 6, 5])   # class attendance gain
gamma = np.array([7, 6, 7, 8])   # assignment-work gain

# assignment completion per hour
sigma = np.array([1, 1, 1, 1])

# min class attendance hours required perweek
L = np.array([3, 2, 3, 2])

# Assignment work required (hours)
A = np.array([4, 3, 4, 3])

# Due days from the sample schedule
# Wed = EOSC, Fri = HISTORY, Sat = MATH2, Sun = MATH1
# Monday=0, Tuesday=1, ..., Sunday=6
D = np.array([
    6,  # MATH1 due Sunday
    4,  # HISTORY due Friday
    5,  # MATH2 due Saturday
    2   # EOSC due Wednesday
])

P = np.array([12, 8, 10, 6])   # example values

G = np.array([250, 180, 200, 160])   # example values

# class schedule a[i,j,k]
# a[i,j,k] = 1 if class is scheduled for course i at hour j on day k
a = np.zeros((nI, nJ, nK), dtype=int)

def add_class(course, day, start_hour, end_hour):
    """
    Add class from start_hour to end_hour (end exclusive)
    Example: 9 to 10 means one hour block at hour 9
    """
    i = course_to_idx[course]
    k = day_to_idx[day]
    for h in range(start_hour, end_hour):
        a[i, h, k] = 1

# MON
add_class("MATH1", "Mon", 9, 10)
add_class("HISTORY","Mon", 10, 11)
add_class("MATH2","Mon", 11, 12)

# WED
add_class("MATH1", "Wed", 9, 10)
add_class("HISTORY","Wed", 10, 11)
add_class("MATH2", "Wed", 11, 12)

# FRI
add_class("MATH1", "Fri", 9, 10)
add_class("MATH2", "Fri", 11, 12)

# THU
add_class("EOSC", "Thu", 11, 15)

# Fixed rest / sleep slots from picture
# rest_fixed[j,k] = 1 if slot is forced to be rest/sleep
rest_fixed = np.zeros((nJ, nK), dtype=int)

# Sleep every day: 00:00-08:00
for k in K:
    for h in range(0,8):
        rest_fixed[h, k] = 1

# Rest every day: 17:00-19:00
for k in K:
    for h in range(17,19):
        rest_fixed[h, k] = 1

# Total weekly hours
H = 24 * 7

# Decision variables
x = cp.Variable((nI, nJ, nK), boolean=True)   # self-study
y = cp.Variable((nI, nJ, nK), boolean=True)   # class attendance
z = cp.Variable((nI, nJ, nK), boolean=True)   # assignment work
s = cp.Variable((nJ, nK), boolean=True)       # rest

## Assumptions

TODO!

## Constraints

In [17]:
constraints = []

### 1. Class Schedule Constraint

A student can only attend a class when the course is actually scheduled. Therefore, class attendance must not exceed the given class timetable.

$$
y_{ijk} \le a_{ijk} \quad , \forall i \in I, j \in J, k \in K
$$

In [18]:
for i in I:
    for j in J:
        for k in K:
            constraints.append(y[i,j,k] <= a[i,j,k])

### 2. Minimum Class Attendance

To ensure that the student attends at least a minimum number of class hours per week for each course, we have the following constraint:

$$
\sum_{j \in J} \sum_{k \in K} y_{ijk} \ge L_i \quad , \forall i \in I
$$

In [19]:
for i in I:
    constraints.append(cp.sum(y[i, :, :]) >= L[i])

### 3. Due day Constraint
To make sure the student can complete the assignment before the due date, we have:

$$
\sum_{j\in J}\sum_{k\in [1,D_i]} \sigma_i z_{ijk} \ge A_i \quad , \forall i \in I 
$$

In [20]:
for i in I:
    due_day = D[i]
    constraints.append(
        sigma[i] * cp.sum(z[i, :, 0:due_day+1]) >= A[i]
    )


### 4. All hours should be scheduled
To make all hours, H (168h), are be scheduled with self-study, having class, do assignment, and take rest, we come with following constraint:
$$
\sum_{i \in I} \sum_{j \in J} \sum_{k \in K} x_{ijk} + y_{ijk} + z_{ijk} + s_{ijk} = H
$$


In [21]:
for j in J:
    for k in K:
        constraints.append(
            cp.sum(x[:, j, k]) + cp.sum(y[:, j, k]) + cp.sum(z[:, j, k]) + s[j, k] == 1
        )

### 5. Fixed Rest and Sleep Hours

Certain time slots are predetermined as rest or sleep periods according to the weekly schedule. During these hours, no studying, class attendance, or assignment work can occur.

$$
s_{jk} = 1
$$

$$
\sum_{i \in I} x_{ijk} = 0
$$

$$
\sum_{i \in I} y_{ijk} = 0
$$

$$
\sum_{i \in I} z_{ijk} = 0
$$

In [22]:
for j in J:
    for k in K:
        if rest_fixed[j,k] == 1:
            constraints.append(s[j,k] == 1)
            constraints.append(cp.sum(x[:,j,k]) == 0)
            constraints.append(cp.sum(y[:,j,k]) == 0)
            constraints.append(cp.sum(z[:,j,k]) == 0)

### 6. Prerequisite learning gain for doing assignment

The student needs to obtain a required amount of learning gain of the certain course $i$ before doing the assignment:

$$
\sum_{k' = 1}^{k-1} \sum_{j' \in J} (\alpha_i x_{ij'k'} + \beta_i y_{ij'k'}) + \sum_{j' = 0}^{j-1} (\alpha_i x_{ij'k} + \beta_i y_{ij'k}) \ge P_i z_{ijk}, \quad \forall i \in I, j \in J, k\in K
$$

In [23]:
for i in I:
    for k in K:
        for j in J:

            prior_learning = 0

            # learning accumulated in previous days
            if k > 0:
                prior_learning += cp.sum(
                    alpha[i] * x[i, :, 0:k] +
                    beta[i]  * y[i, :, 0:k]
                )

            # learning accumulated earlier on the same day
            if j > 0:
                prior_learning += cp.sum(
                    alpha[i] * x[i, 0:j, k] +
                    beta[i]  * y[i, 0:j, k]
                )

            constraints.append(
                prior_learning >= P[i] * z[i, j, k]
            )

### 7. Maximum learning gain constraint

Each course have a maximum learning gain which can be obtained from study behaviors.

$$
\sum_{j \in J} \sum_{k \in K}
\left(
\alpha_i x_{ijk} + \beta_i y_{ijk} + \gamma_i z_{ijk}
\right) \le G_i , \quad \forall i \in I
$$

In [24]:
for i in I:
    constraints.append(
        cp.sum(
            alpha[i] * x[i, :, :] +
            beta[i]  * y[i, :, :] +
            gamma[i] * z[i, :, :]
        ) <= G[i]
    )

## Objective Function

The goal is to maximize the total learning gain obtained from self-study, attending classes, and completing assignments across all courses and time periods.

$$
\max \sum_{i \in I} \sum_{j \in J} \sum_{k \in K}
\left(
\alpha_i x_{ijk} + \beta_i y_{ijk} + \gamma_i z_{ijk}
\right)
$$

In [25]:
objective = cp.Maximize(
    cp.sum(cp.multiply(alpha[:, None, None], x)) +
    cp.sum(cp.multiply(beta[:, None, None], y)) +
    cp.sum(cp.multiply(gamma[:, None, None], z))
)

## Solving the model

In [26]:
problem = cp.Problem(objective, constraints)
problem.solve()

print("Status:", problem.status)
print("Optimal learning gain:", problem.value)

/opt/miniconda3/envs/cpsc330/lib/python3.12/site-packages/cvxpy/reductions/solvers/solving_chain_utils.py:30: UserWarning: The problem includes expressions that don't support CPP backend. Defaulting to the SCIPY backend for canonicalization.
  warnings.warn(UserWarning(


Status: optimal
Optimal learning gain: 730.0


In [27]:
def activity_label(j, k):
    # check rest
    if s.value[j, k] > 0.5:
        return "REST"

    # check course activities
    for i, c in enumerate(courses):
        if x.value[i, j, k] > 0.5:
            return f"Study-{c}"
        if y.value[i, j, k] > 0.5:
            return f"Class-{c}"
        if z.value[i, j, k] > 0.5:
            return f"Assign-{c}"

    return "UNASSIGNED"

schedule = pd.DataFrame(index=[f"{h:02d}:00" for h in hours], columns=days)

for k, day in enumerate(days):
    for j, h in enumerate(hours):
        schedule.loc[f"{h:02d}:00", day] = activity_label(j, k)

print("\nWeekly Optimal Schedule:")
print(schedule)


Weekly Optimal Schedule:
                 Mon            Tue            Wed             Thu  \
00:00           REST           REST           REST            REST   
01:00           REST           REST           REST            REST   
02:00           REST           REST           REST            REST   
03:00           REST           REST           REST            REST   
04:00           REST           REST           REST            REST   
05:00           REST           REST           REST            REST   
06:00           REST           REST           REST            REST   
07:00           REST           REST           REST            REST   
08:00    Study-MATH1  Study-HISTORY    Study-MATH2     Study-MATH1   
09:00    Class-MATH1    Study-MATH1    Class-MATH1   Study-HISTORY   
10:00  Class-HISTORY    Assign-EOSC  Class-HISTORY     Study-MATH2   
11:00    Class-MATH2    Study-MATH1    Class-MATH2      Class-EOSC   
12:00    Study-MATH1  Study-HISTORY    Study-MATH1     Assign-EO

In [28]:
# summary by course
summary_rows = []
for i, c in enumerate(courses):
    study_hours = int(np.round(np.sum(x.value[i, :, :])))
    class_hours = int(np.round(np.sum(y.value[i, :, :])))
    assign_hours = int(np.round(np.sum(z.value[i, :, :])))
    summary_rows.append([c, study_hours, class_hours, assign_hours])

summary = pd.DataFrame(summary_rows, columns=["Course", "SelfStudyHours", "ClassHours", "AssignmentHours"])
print("\nSummary:")
print(summary)


Summary:
    Course  SelfStudyHours  ClassHours  AssignmentHours
0    MATH1              25           3                4
1  HISTORY              14           2                3
2    MATH2              19           3                4
3     EOSC               1           2               18
